# Data Preprocessing for VLST Dataset

## Overview

This notebook prepares the VLST (Very Late Stent Thrombosis) data for modeling, based on findings from the EDA. It implements a reproducible pipeline that:

1. **Loads raw data** and drops non-predictive columns
2. **Handles missing values** (imputation)
3. **Encodes categorical variables**
4. **Splits data** into train/test (stratified)
5. **Transforms continuous features** (scaling)
6. **Saves preprocessed datasets and artifacts** for the modeling phase

### EDA Summary (from eda.ipynb)

- **Target**: `Stent thrombosis` (binary: 0/1)
- **Features**: Binary, continuous, and one categorical (stent type)
- **Paths**: Raw data `../data/raw/VLST.csv`; outputs to `../data/processed/`

## 1. Setup and Imports

In [2]:
import pandas as pd
import numpy as np
import os
import joblib
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
np.random.seed(42)

# Paths (run from code/)
RAW_PATH = "../data/raw/VLST.csv"
OUT_DIR = "../data/processed"
os.makedirs(OUT_DIR, exist_ok=True)

TARGET_COL = "Stent thrombosis"
print(f"✓ Raw data: {RAW_PATH}")
print(f"✓ Output dir: {OUT_DIR}")

✓ Raw data: ../data/raw/VLST.csv
✓ Output dir: ../data/processed


## 2. Load Raw Data and Drop Non-Predictive Columns

We remove identifier columns (`NO.`, `Name`) that must not be used as predictors.

In [3]:
df = pd.read_csv(RAW_PATH)

# Drop identifier columns (not used as features)
drop_cols = [c for c in ["NO.", "Name"] if c in df.columns]
df = df.drop(columns=drop_cols, errors="ignore")

print(f"Shape after dropping IDs: {df.shape}")
print(f"Target distribution:\n{df[TARGET_COL].value_counts()}")
df.head(3)

Shape after dropping IDs: (5185, 83)
Target distribution:
Stent thrombosis
0    5093
1      92
Name: count, dtype: int64


,Stent thrombosis,Age,Men,Time since stent implantation,Diabetes,Hypertension,HL,Current smoker,Current drinking,Stroke/TIA,...,LDL,HDL,TG,Fast-Glu,HbA1c,Fiberinogen,Aspirin,Clopidogrel,Ticagrelor,DAPT
0,0,36,1,1243,1,1,0,1,1,0,...,2.60,0.87,4.20,7.64,6.8,2.14,1,0,0,0
1,0,47,1,1540,0,0,0,1,0,0,...,1.41,1.88,1.37,4.99,4.6,1.71,1,1,0,1
2,0,45,1,1567,1,0,1,1,0,0,...,2.71,1.03,2.38,8.10,9.3,3.01,1,1,0,1


## 3. Feature Categorization

Separate columns into numeric (binary + continuous) and categorical, consistent with EDA.

In [4]:
# All feature columns (exclude target)
feature_cols = [c for c in df.columns if c != TARGET_COL]
numeric_cols = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df[feature_cols].select_dtypes(include=["object"]).columns.tolist()

# Binary: numeric with only 0/1 (and possibly NaN)
binary_cols = []
for c in numeric_cols:
    uniq = df[c].dropna().unique()
    if len(uniq) <= 2 and set(uniq).issubset({0, 1}):
        binary_cols.append(c)
continuous_cols = [c for c in numeric_cols if c not in binary_cols]

print(
    f"Binary: {len(binary_cols)}, Continuous: {len(continuous_cols)}, Categorical: {len(cat_cols)}"
)
print(f"Categorical columns: {cat_cols}")

Binary: 57, Continuous: 24, Categorical: 1
Categorical columns: ['Stent type-SES']


## 4. Missing Value Summary

Before imputation, we report missing counts per column (EDA-informed strategy: impute continuous with median, binary/categorical with mode).

In [5]:
missing = df[feature_cols].isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if len(missing) > 0:
    print("Columns with missing values:")
    print(missing)
else:
    print("No missing values in feature columns.")

No missing values in feature columns.


## 5. Build Preprocessing Pipeline

- **Continuous**: Impute with median, then standardize (fit on train later).
- **Binary**: Impute with mode (most frequent 0/1).
- **Categorical**: Impute with mode, then one-hot encode (drop first to avoid collinearity in linear models).

In [6]:
# Imputation + scaling/encoding (numeric scaled in step 7 after split)
imputer_cont = SimpleImputer(strategy="median")
imputer_bin = SimpleImputer(strategy="most_frequent")
imputer_cat = SimpleImputer(strategy="constant", fill_value="missing")
ohe = OneHotEncoder(drop="first", handle_unknown="ignore")

# ColumnTransformer for imputation + encoding (no scaling yet)
transformers = []
if continuous_cols:
    transformers.append(("num", imputer_cont, continuous_cols))
if binary_cols:
    transformers.append(("bin", imputer_bin, binary_cols))
if cat_cols:
    transformers.append(
        ("cat", Pipeline([("impute", imputer_cat), ("ohe", ohe)]), cat_cols)
    )

preprocessor = ColumnTransformer(
    transformers, remainder="drop", verbose_feature_names_out=False
)
print("Preprocessor (imputation + OHE for categorical):")
print(preprocessor)

Preprocessor (imputation + OHE for categorical):
ColumnTransformer(transformers=[('num', SimpleImputer(strategy='median'),
                                 ['Age', 'Time since stent implantation',
                                  'NO.of vessels', 'Min-stent diameter',
                                  'Max-stent diameter', 'Total stent length',
                                  'Stent release pressure',
                                  'No.of stents per lesion', 'LV', 'LVEF',
                                  'CaI', 'WBC', 'HGB', 'Platelet', 'Cre',
                                  'eGFR', 'CKD5', 'TCL', 'LDL', 'HDL', 'TG',
                                  'Fast-Glu', 'HbA1c', 'Fiberinogen']),
                                ('b...
                                  '3-vessel disease', 'Chronic total occlusion',
                                  'Moderate/severe calcification',
                                  'Moderate/severe tortuosity',
                                  'Lesion l

## 5b. (Optional) Drop Constant or Near-Constant Features

Features with zero or very low variance add no information and can cause issues in some models.

In [7]:
# Optional: drop constant columns before split (no variance)
constant_cols = [c for c in feature_cols if df[c].nunique(dropna=False) <= 1]
if constant_cols:
    print(f"Dropping constant columns: {constant_cols}")
    feature_cols = [c for c in feature_cols if c not in constant_cols]
    binary_cols = [c for c in binary_cols if c not in constant_cols]
    continuous_cols = [c for c in continuous_cols if c not in constant_cols]
else:
    print("No constant columns to drop.")

No constant columns to drop.


## 6. Train/Test Split (Stratified)

Stratify by target to preserve class proportion. Default 80/20 split.

In [8]:
X = df[feature_cols].copy()
y = df[TARGET_COL].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")
print(f"Train target: {y_train.value_counts().to_dict()}")
print(f"Test target:  {y_test.value_counts().to_dict()}")

Train: 4148, Test: 1037
Train target: {0: 4074, 1: 74}
Test target:  {0: 1019, 1: 18}


## 7. Fit Preprocessor on Train and Transform

Imputation and one-hot encoding are fitted on the training set only, then applied to both train and test.

In [9]:
# Fit on train, transform both
X_train_enc = preprocessor.fit_transform(X_train)
X_test_enc = preprocessor.transform(X_test)

# Get feature names after preprocessing
feature_names_out = preprocessor.get_feature_names_out()
print(f"Feature matrix shape - Train: {X_train_enc.shape}, Test: {X_test_enc.shape}")
print(f"Feature names (first 15): {list(feature_names_out[:15])}")

Feature matrix shape - Train: (4148, 174), Test: (1037, 174)
Feature names (first 15): ['Age', 'Time since stent implantation', 'NO.of vessels', 'Min-stent diameter', 'Max-stent diameter', 'Total stent length', 'Stent release pressure', 'No.of stents per lesion', 'LV', 'LVEF', 'CaI', 'WBC', 'HGB', 'Platelet', 'Cre']


## 8. Scale Features (Fit on Train)

StandardScaler is fitted on the training set only to avoid data leakage.

In [10]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_enc)
X_test_scaled = scaler.transform(X_test_enc)

# Preprocessed arrays (float, ready for modeling)
print(f"Final shapes - Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")
print(f"Train sample (first row, first 5): {X_train_scaled[0, :5]}")

Final shapes - Train: (4148, 174), Test: (1037, 174)
Train sample (first row, first 5): [-0.89654786 -0.80211523 -1.05969933  1.28186295  1.09850741]


## 9. Save Preprocessed Data and Artifacts

Save NumPy arrays (X_train, X_test, y_train, y_test), feature names, and fitted preprocessor + scaler for the modeling phase.

In [11]:
# Save arrays
np.save(os.path.join(OUT_DIR, "X_train.npy"), X_train_scaled)
np.save(os.path.join(OUT_DIR, "X_test.npy"), X_test_scaled)
np.save(os.path.join(OUT_DIR, "y_train.npy"), y_train.values)
np.save(os.path.join(OUT_DIR, "y_test.npy"), y_test.values)

# Feature names
pd.Series(feature_names_out).to_csv(
    os.path.join(OUT_DIR, "feature_names.csv"), index=False, header=["feature_name"]
)

# Fitted objects (for inference / reproducibility)
joblib.dump(preprocessor, os.path.join(OUT_DIR, "preprocessor.joblib"))
joblib.dump(scaler, os.path.join(OUT_DIR, "scaler.joblib"))

# Column role config (for reproducibility / reporting)
pd.concat(
    [
        pd.DataFrame({"role": "binary", "column": binary_cols}),
        pd.DataFrame({"role": "continuous", "column": continuous_cols}),
        pd.DataFrame({"role": "categorical", "column": cat_cols}),
    ],
    ignore_index=True,
).to_csv(os.path.join(OUT_DIR, "column_config.csv"), index=False)

# Also save as DataFrames for convenience (with target)
train_df = pd.DataFrame(X_train_scaled, columns=feature_names_out)
train_df[TARGET_COL] = y_train.values
test_df = pd.DataFrame(X_test_scaled, columns=feature_names_out)
test_df[TARGET_COL] = y_test.values
try:
    train_df.to_parquet(os.path.join(OUT_DIR, "train.parquet"), index=False)
    test_df.to_parquet(os.path.join(OUT_DIR, "test.parquet"), index=False)
except Exception:
    train_df.to_csv(os.path.join(OUT_DIR, "train.csv"), index=False)
    test_df.to_csv(os.path.join(OUT_DIR, "test.csv"), index=False)

print("Saved to", OUT_DIR)
for f in sorted(os.listdir(OUT_DIR)):
    p = os.path.join(OUT_DIR, f)
    print(f"  - {f} ({os.path.getsize(p) / 1024:.1f} KB)")

Saved to ../data/processed
  - X_test.npy (1409.8 KB)
  - X_train.npy (5638.8 KB)
  - column_config.csv (1.6 KB)
  - feature_names.csv (3.4 KB)
  - preprocessor.joblib (8.8 KB)
  - scaler.joblib (4.6 KB)
  - test.csv (3661.4 KB)
  - train.csv (14636.5 KB)
  - y_test.npy (8.2 KB)
  - y_train.npy (32.5 KB)


## 10. Quick Verification

Load back and check shapes and target balance.

In [12]:
# Reload and verify
X_tr = np.load(os.path.join(OUT_DIR, "X_train.npy"))
X_te = np.load(os.path.join(OUT_DIR, "X_test.npy"))
y_tr = np.load(os.path.join(OUT_DIR, "y_train.npy"))
y_te = np.load(os.path.join(OUT_DIR, "y_test.npy"))
names = pd.read_csv(os.path.join(OUT_DIR, "feature_names.csv"))["feature_name"].tolist()

assert X_tr.shape[1] == len(names) == X_te.shape[1]
assert np.isnan(X_tr).sum() == 0 and np.isnan(X_te).sum() == 0
print("✓ Shapes and feature count consistent")
print("✓ No NaN in preprocessed arrays")
print(
    f"✓ Ready for modeling: {X_tr.shape[0]} train, {X_te.shape[0]} test, {len(names)} features"
)

✓ Shapes and feature count consistent
✓ No NaN in preprocessed arrays
✓ Ready for modeling: 4148 train, 1037 test, 174 features


## 11. Summary

Preprocessing is complete. For modeling you can:

- **NumPy**: `np.load('data/processed/X_train.npy')`, same for `X_test`, `y_train`, `y_test`.
- **DataFrames**: `pd.read_parquet('data/processed/train.parquet')` (or `train.csv` if parquet unavailable), same for test.
- **Feature names**: `pd.read_csv('data/processed/feature_names.csv')`.
- **Column roles**: `pd.read_csv('data/processed/column_config.csv')`.
- **Inference**: Load `preprocessor.joblib` and `scaler.joblib` to transform new raw data the same way.

**Note on class imbalance**: EDA showed strong imbalance. Consider class weights or resampling (e.g. SMOTE) in the modeling phase.